# 06 · Evaluate and analyse  (steps 9, 10, 13, 14)
**Accelerator: None (CPU, uses no GPU quota).** Run with **Save Version → Save & Run All (Commit)**; you can close the laptop. Re-running continues where the last run stopped, on any account.


Scores every finished run that has no scores yet (dev/ablation runs on val, final runs on test), uploads the
scores, builds the tables, the BatchNorm-check comparison, the agreement-bin analysis and the efficiency
table. Safe to re-run at any time.

In [ ]:
# ==== SETTINGS (nothing here needs to change between Kaggle accounts) ====
HF_REPO = "sujalprasad90/pedger-mtkd-store"   # the ONE private Hugging Face repo shared by every account
ACCOUNT = ""                                  # optional label, e.g. your name; sessions are told apart automatically
REPO_URL = ""                                 # optional GitHub URL of the code ("" = Kaggle dataset or HF copy)
# Once per Kaggle account, and for every imported notebook:
#   Add-ons -> Secrets -> HF_TOKEN = a WRITE token from the Hugging Face account that owns HF_REPO, ticked for this notebook
#   Settings -> Internet: On;  Settings -> Accelerator: as written in the title
# The preflight cell below checks all of this and stops with a plain-language fix if something is missing.

In [ ]:
# ---- setup: secrets, code, dependencies (no edits needed) ----
import glob, os, shutil, subprocess, sys, time
SESSION_START = time.time()
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Secret HF_TOKEN is not available: Add-ons -> Secrets -> add HF_TOKEN and tick it for THIS "
                       "notebook, then run again.") from e
os.environ["PEDGER_REMOTE"] = "hf:" + HF_REPO
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.25,<2.0", "pyyaml"], check=True)
W, REPO, WS = "/kaggle/working", "/kaggle/working/pedger-mtkd", "/kaggle/working/ws"
if not os.path.exists(REPO):
    attached = sorted(glob.glob("/kaggle/input/*/pedger_kd/__init__.py") + glob.glob("/kaggle/input/*/*/pedger_kd/__init__.py"))
    if attached:                                   # code uploaded as a Kaggle Dataset (pedger-mtkd-code)
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), REPO)
        CODE_SOURCE = "kaggle dataset"
    elif REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
        CODE_SOURCE = "git"
    else:                                          # copy pushed to Hugging Face by notebook 01
        from huggingface_hub import snapshot_download
        snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["code/*"], local_dir=f"{W}/hf_code",
                          token=os.environ["HF_TOKEN"])
        shutil.copytree(f"{W}/hf_code/code", REPO)
        CODE_SOURCE = "huggingface"
    print("code from:", CODE_SOURCE)
os.chdir(REPO)
sys.path.insert(0, REPO)
!python tools/fetch_upstream.py
# preflight: session type, GPUs, internet, Hugging Face account and repo access
if subprocess.run([sys.executable, "-m", "pedger_kd.preflight"]).returncode:
    raise RuntimeError("Preflight failed: fix the PROBLEM lines printed above, then run the notebook again.")

In [ ]:
!pip install -q -r requirements.txt
!python tools/install_pyedgeeval.py   # BSDS benchmark scorer; patched build for Python 3.13+, ~1-2 min

In [ ]:
# BSDS500 original release (per-annotator ground truth); ~70 MB
RAW_BSDS = "/kaggle/working/BSR/BSDS500/data"
if not os.path.exists(RAW_BSDS):
    !wget -q https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/BSR/BSR_bsds500.tgz -O /kaggle/working/bsr.tgz
    !tar -xzf /kaggle/working/bsr.tgz -C /kaggle/working && rm /kaggle/working/bsr.tgz
print(sorted(os.listdir(RAW_BSDS + "/images")))

In [ ]:
!python eval/eval_runs.py --remote hf:{HF_REPO} --ws {WS} --raw-root bsds={RAW_BSDS} --nproc 4
# multi-scale for the main rows only:
# !python eval/eval_runs.py --remote hf:{HF_REPO} --ws {WS} --raw-root bsds={RAW_BSDS} --nproc 4 --ms --runs "*_s?" 

In [ ]:
!python results/make_tables.py --scores "{WS}/runs/*/scores_*.json" --out {W}/results/tables.md
!python -m pedger_kd.sync push-dir --local {W}/results --to results

In [ ]:
# step 9: BatchNorm check (needs the three bn_* runs scored on val)
r = f"{WS}/runs"
if all(os.path.exists(f"{r}/{c}_s0/scores_val_ns_m.json") for c in ["bn_micro1", "bn_micro4", "bn_stock4"]):
    !python eval/bn_check.py --compare-runs {r}/bn_micro1_s0/scores_val_ns_m.json {r}/bn_micro4_s0/scores_val_ns_m.json {r}/bn_stock4_s0/scores_val_ns_m.json
else:
    print("BatchNorm-check runs not finished/scored yet")

In [ ]:
# step 13: agreement bins (test split, seed 0)
runs = {"PEdger": "baseline_s0", "A_uniform": "A_uniform_s0", "Ours": "A_agree_s0"}
args = " ".join(f"--run {k}={WS}/runs/{v}/scores_test_ns_m.json" for k, v in runs.items()
                if os.path.exists(f"{WS}/runs/{v}/scores_test_ns_m.json"))
if args.count("--run") >= 2:
    from pedger_kd.sync import open_remote
    remote = open_remote(os.environ["PEDGER_REMOTE"])
    for v in runs.values():
        remote.download(f"runs/{v}/pred_test_ns_m", WS)
    !python -m pedger_kd.sync pull --prefix data/bsds/eval_lists --ws {WS}
    !python eval/agreement_bins.py --store {WS}/data/bsds --split test --raw-root {RAW_BSDS} {args} --out {W}/results/agreement_bins.json
    !python -m pedger_kd.sync push-dir --local {W}/results --to results
else:
    print("need at least two scored test runs")

In [ ]:
# step 14: efficiency + training cost (CPU FPS here; run bench/speed_flops.py in a GPU notebook for T4 FPS)
TEACHER_HOURS = 0.0   # GPU-hours printed by notebook 03
!python bench/speed_flops.py --size 320 480 --runs "{WS}/runs/*" --teacher-hours {TEACHER_HOURS} --out {W}/results/efficiency.json
!python -m pedger_kd.sync push-dir --local {W}/results --to results

In [ ]:
# keep the notebook output small (everything important is already on Hugging Face)
for d in ["ws/data", "ws/teachers", "ws/.staging", "hf_code", "work", "BSR"]:
    shutil.rmtree(f"/kaggle/working/{d}", ignore_errors=True)
print(f"session used {(time.time() - SESSION_START) / 3600:.2f} h")